[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bsheese/cs377/blob/main/supplemental/Ridge_vs_Lasso_Pair.ipynb)

# Ridge vs. Lasso: Who Gets the Credit?
**CS 377 · companion to 17_2_1_3 Regularization · pair activity**

**Driver:** this notebook is yours. Your partner, the **Recorder**, has the **Ridge vs. Lasso quiz in Canvas** open, reads each question aloud and types your agreed answers.

**How the two fit together.** The quiz is the guide; this notebook is where you do the work for the quiz questions tagged **💻 NOTEBOOK**. Each section below is named after its quiz question (**Q6** here is **Q6** in the quiz). Questions tagged **💬 NO NOTEBOOK** have no section here: just talk them through.

**For each notebook question:** the Recorder reads it aloud; you find the matching section, write the code in the cell marked **✏️ YOUR CODE**, and run it (Shift+Enter). Read the result aloud and agree with your partner on what it means. Hints are under each section: try without them first, and open Hint 1 before Hint 2.

## Setup (run this first)
**Part 1 of the quiz (Q1-Q5) has no notebook work.** Run this cell while the Recorder reads Part 1; loading the data takes a few seconds.

It loads the same Ames training set as 17_2_1_3 and defines two helpers you will use in every section:
- `make_pipe(model)`: a `StandardScaler` followed by the model, so every feature is standardized before the penalty.
- `coefs(model)`: fits `make_pipe(model)` on the training set and returns all 225 coefficients as a Series, one per feature name. `coefs(model, X, y)` fits on other data instead.

In [ ]:
import warnings
warnings.filterwarnings('ignore')   # Lasso prints convergence warnings for small alphas

import urllib.request
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.model_selection import cross_val_score

# the Ames data, cleaned exactly as in 17_2_1_1 (same training set as 17_2_1_3)
module_url = "https://raw.githubusercontent.com/bsheese/cs377/main/17_regression_crossval/17_2_MLR/ames_cleaning.py"
urllib.request.urlretrieve(module_url, "ames_cleaning.py")
from ames_cleaning import load_and_clean_ames

data_url = "https://raw.githubusercontent.com/bsheese/cs377/main/data/data_housing_ames.txt"
X_train, X_test, y_train, y_test = load_and_clean_ames(data_url)   # y is log sale price


def make_pipe(model):
    """StandardScaler, then the model (the same pipeline as 17_2_1_3)."""
    return Pipeline([('scaler', StandardScaler()), ('regressor', model)])


def coefs(model, X=X_train, y=y_train):
    """Fit the scaler + model on X, y and return the coefficients as a Series indexed by feature name."""
    pipe = make_pipe(model).fit(X, y)
    return pd.Series(pipe.named_steps['regressor'].coef_, index=X.columns)


print("training houses:", X_train.shape[0], "  features:", X_train.shape[1])

# example: two of Ridge's 225 coefficients at alpha = 100
coefs(Ridge(alpha=100))[['Overall Qual', 'Garage Cars']]

# Part 2 (Q6-Q11): Real Data: All 225 Ames Features
Sections here: Q6, Q7, Q8, Q9, Q10, Q11.

## Q6 · How many features survive?
> **Quiz Q6:** Fit OLS, Ridge (α = 100) and Lasso (α = 0.005) on all 225 features. How many coefficients does each model keep that are not zero? Which rule from Q4 does this show?

**✏️ Your code:** For each of the three models, count the coefficients that are not zero. The `coefs()` helper from Setup fits the model (with the scaler) and gives back all 225 coefficients.

**Tell your Recorder:** the three counts, then your answer.

<details><summary>Hint 1</summary>

`coefs(Ridge(alpha=100))` is a Series of 225 numbers. `!= 0` turns it into `True`/`False`, and `.sum()` counts the `True`s.

</details>

<details><summary>Hint 2</summary>

Write one line per model, or loop over a list of `(name, model)` pairs and `print` each count.

</details>

In [ ]:
# YOUR CODE: count the nonzero coefficients for
#   LinearRegression(), Ridge(alpha=100) and Lasso(alpha=0.005)

## Q7 · Turn the dial
> **Quiz Q7:** Turn the dial. Count the nonzero coefficients for Lasso at α = 0.001, 0.005, 0.02, 0.05 and 0.1, and for Ridge at α = 10, 100, 1,000 and 10,000. Describe each model's pattern. Does Ridge ever drop a feature?

**✏️ Your code:** Loop over each list of alphas (already typed in for you) and print how many coefficients are not zero at each alpha.

**Tell your Recorder:** the Lasso count at α = 0.1, then one sentence on each model's pattern.

<details><summary>Hint 1</summary>

Same count as Q6, inside a `for a in lasso_alphas:` loop, using `Lasso(alpha=a)`.

</details>

<details><summary>Hint 2</summary>

Then a second loop for `ridge_alphas` with `Ridge(alpha=a)`.

</details>

In [ ]:
lasso_alphas = [0.001, 0.005, 0.02, 0.05, 0.1]
ridge_alphas = [10, 100, 1000, 10000]

# YOUR CODE: for each alpha, print the alpha and the number of nonzero coefficients

## Q8 · The garage twins
> **Quiz Q8:** The garage twins. Check the correlation between `Garage Cars` and `Garage Area`, then compare their two coefficients in OLS, Ridge (α = 1,000) and Lasso (α = 0.1). What did each model do to the pair? Was your Q5 prediction right?

**✏️ Your code:** Print the correlation between the two garage columns, then put the two garage coefficients from each of the three models side by side.

**Tell your Recorder:** a few words for what each model did to the pair, then your answer.

<details><summary>Hint 1</summary>

`X_train[twins].corr()` gives the correlation table. `coefs(model)[twins]` picks just the two garage coefficients.

</details>

<details><summary>Hint 2</summary>

`pd.DataFrame({'OLS': ..., 'Ridge 1000': ..., 'Lasso 0.1': ...})` puts three Series side by side; add `.round(3)`.

</details>

In [ ]:
twins = ['Garage Cars', 'Garage Area']

# YOUR CODE: 1) the correlation between the twins
#   2) the twins' coefficients in OLS, Ridge(alpha=1000) and Lasso(alpha=0.1)

## Q9 · Who survives, and who grows?
> **Quiz Q9:** Lasso at α = 0.1 keeps only six features. Which are they? `Log_Gr Liv Area` was deleted: which survivor is its near-twin? `Overall Qual`'s coefficient is 0.063 in OLS but 0.129 here. How can adding a penalty make a coefficient bigger?

**✏️ Your code:** Keep only the nonzero Lasso (α = 0.1) coefficients, then find which survivor is most correlated with `Log_Gr Liv Area`.

**Tell your Recorder:** the near-twin you found, then your explanation.

<details><summary>Hint 1</summary>

Save `coefs(Lasso(alpha=0.1))` in a variable, then keep the rows where it is `!= 0` (a True/False filter inside `[...]`). Its `.index` holds the survivors' names.

</details>

<details><summary>Hint 2</summary>

`X_train[names].corrwith(X_train['Log_Gr Liv Area'])` gives each survivor's correlation with that one column.

</details>

In [ ]:
# YOUR CODE: 1) the nonzero coefficients of Lasso(alpha=0.1)
#   2) the correlation of each survivor with 'Log_Gr Liv Area'

## Q10 · Same choice every time?
> **Quiz Q10:** Is Lasso's choice between the twins stable? Draw 8 random samples of 40 houses and fit OLS, Ridge (α = 20) and Lasso (α = 0.05) on just the two garage features. In how many samples did Lasso drop one twin, and was it always the same twin? What did OLS and Ridge do across the samples?

**✏️ Your code:** The loop that draws the samples is written for you. Inside it, fit the three models on `sample` and `y_sample` and print the sample number with each model's two coefficients.

**Tell your Recorder:** the count and which twin was dropped each time, then what OLS and Ridge did.

<details><summary>Hint 1</summary>

`coefs()` takes the data as extra arguments: `coefs(Lasso(alpha=0.05), sample, y_sample)`.

</details>

<details><summary>Hint 2</summary>

Add `.round(2).tolist()` to make each pair short, then `print(seed, ...)` the three pairs on one line.

</details>

In [ ]:
twins = ['Garage Cars', 'Garage Area']

for seed in range(8):
    sample = X_train[twins].sample(40, random_state=seed)   # 40 random houses
    y_sample = y_train.loc[sample.index]                      # their log prices
    # YOUR CODE: fit LinearRegression(), Ridge(alpha=20) and Lasso(alpha=0.05) on sample, y_sample
    #   and print seed with each model's two coefficients (round to 2 places)
    pass   # delete this line when you add your code

## Q11 · Does deleting cost accuracy?
> **Quiz Q11:** Does deleting features cost accuracy? Compute the 5-fold cross-validated log-RMSE (lower is better) for OLS, Ridge (α = 100), Lasso (α = 0.005, 66 features) and Lasso (α = 0.02, 26 features). Which models are essentially tied? What happens to the error as Lasso removes more features?

**✏️ Your code:** For each model in `models` (typed in for you), run 5-fold `cross_val_score` on the scaler + model pipeline and print the RMSE, as in 17_2_1_3.

**Tell your Recorder:** which models are tied, then what happened at α = 0.02.

<details><summary>Hint 1</summary>

`cross_val_score(make_pipe(model), X_train, y_train, cv=5, scoring='neg_mean_squared_error')` gives five negative MSEs, one per fold.

</details>

<details><summary>Hint 2</summary>

Take `-scores.mean()` to get the MSE, then `np.sqrt(...)` to get the RMSE.

</details>

In [ ]:
models = {'OLS': LinearRegression(),
          'Ridge 100': Ridge(alpha=100),
          'Lasso 0.005': Lasso(alpha=0.005),
          'Lasso 0.02': Lasso(alpha=0.02)}

# YOUR CODE: for each model, print its 5-fold CV log-RMSE

# Part 3 (Q12-Q16): Choose a Penalty
No notebook sections: these are 💬 NO NOTEBOOK questions. Answer them in the quiz using what you found above.

## Extra (optional): watch the coefficients move
Five coefficients as alpha grows, Ridge on the left and Lasso on the right. Find the garage twins and `Log_Gr Liv Area`: does the picture match what you found in Q7-Q9?

In [ ]:
import matplotlib.pyplot as plt

show = ['Overall Qual', 'Total_Square_Footage', 'Log_Gr Liv Area', 'Garage Cars', 'Garage Area']
lasso_grid = np.logspace(-3.5, -0.7, 25)
ridge_grid = np.logspace(0, 4.5, 25)
lasso_path = pd.DataFrame([coefs(Lasso(alpha=a))[show] for a in lasso_grid], index=lasso_grid)
ridge_path = pd.DataFrame([coefs(Ridge(alpha=a))[show] for a in ridge_grid], index=ridge_grid)

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
ridge_path.plot(ax=axes[0], logx=True, title='Ridge: shrinks, never zero')
lasso_path.plot(ax=axes[1], logx=True, title='Lasso: hits zero, survivors can grow', legend=False)
for ax in axes:
    ax.axhline(0, color='gray', lw=0.8)
    ax.set_xlabel('alpha (log scale)')
axes[0].set_ylabel('standardized coefficient')
plt.tight_layout()
plt.show()